# repo-locate × FastContext — Kaggle integration test

This is the Kaggle counterpart to the Colab integration test. It uses the same public synthetic fixture and oracle; no private project or competition material is used.

**Before running:** enable **Internet** and choose **GPU T4 x2** in Kaggle settings. Then run all cells.

In [ ]:
import subprocess, sys, os
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import os, pathlib, shutil, subprocess

# Install uv without assuming where the installer puts it.
subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
candidates = [shutil.which('uv'), '/usr/local/bin/uv', '/root/.local/bin/uv', str(pathlib.Path.home()/'.local/bin/uv')]
UV = next((p for p in candidates if p and pathlib.Path(p).exists()), None)
assert UV, f'uv not found; checked {candidates}'
print('uv:', UV)

ROOT = '/kaggle/working'
VLLM_ENV = f'{ROOT}/vllm-env'
FC_ENV = f'{ROOT}/fastcontext-env'
VLLM = f'{VLLM_ENV}/bin/vllm'
FASTCONTEXT = f'{FC_ENV}/bin/fastcontext'

subprocess.run([UV, 'python', 'install', '3.12'], check=True)
for env_dir in (VLLM_ENV, FC_ENV):
    shutil.rmtree(env_dir, ignore_errors=True)
    subprocess.run([UV, 'venv', '--python', '3.12', env_dir], check=True)

subprocess.run([UV, 'pip', 'install', '--python', f'{VLLM_ENV}/bin/python', 'vllm', '--torch-backend=auto'], check=True)
subprocess.run([UV, 'pip', 'install', '--python', f'{FC_ENV}/bin/python', 'git+https://github.com/Alssndr0/fastcontext.git'], check=True)

shutil.rmtree(f'{ROOT}/repo-locate', ignore_errors=True)
subprocess.run(['git','clone','-q','https://github.com/janposlusny/repo-locate.git',f'{ROOT}/repo-locate'], check=True)

assert pathlib.Path(VLLM).exists()
assert pathlib.Path(FASTCONTEXT).exists()
print('vLLM:', VLLM)
print('fastcontext:', FASTCONTEXT)
subprocess.run([f'{VLLM_ENV}/bin/python','-c',"import torch; print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)"], check=True)


In [ ]:
import pathlib, requests, subprocess, time, os

MODEL_ID = 'microsoft/FastContext-1.0-4B-SFT'
SERVED_NAME = 'qwen3-fastcontext-sft'
LOG = '/kaggle/working/vllm-fastcontext.log'

server_env = os.environ.copy()
server_env['CUDA_VISIBLE_DEVICES'] = '0'
log = open(LOG, 'w')
server = subprocess.Popen([
    VLLM, 'serve', MODEL_ID,
    '--served-model-name', SERVED_NAME,
    '--dtype', 'float16',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.92',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
    '--enable-prefix-caching',
    '--enforce-eager',
    '--host', '127.0.0.1', '--port', '8000',
], env=server_env, stdout=log, stderr=subprocess.STDOUT)
print('vLLM PID:', server.pid)

for _ in range(180):
    if server.poll() is not None:
        log.flush(); print(pathlib.Path(LOG).read_text()[-12000:]); raise RuntimeError('vLLM exited before becoming ready')
    try:
        r = requests.get('http://127.0.0.1:8000/v1/models', timeout=2)
        if r.ok:
            print('FastContext endpoint ready:', r.json()['data'][0]['id'])
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    log.flush(); print(pathlib.Path(LOG).read_text()[-12000:]); raise TimeoutError('vLLM did not become ready')


In [ ]:
import os, subprocess
env = os.environ.copy()
env['PATH'] = f'{FC_ENV}/bin:' + env['PATH']
env.update({
    'BASE_URL':'http://127.0.0.1:8000/v1',
    'MODEL':'qwen3-fastcontext-sft',
    'API_KEY':'local',
    'FASTCONTEXT_MAX_TURNS':'8',
    'FASTCONTEXT_MAX_TOKENS':'4000',
    'FASTCONTEXT_TRAJ_DIR':'/kaggle/working/fastcontext-trajectories',
})

result = subprocess.run([
    f'{FC_ENV}/bin/python', f'{ROOT}/repo-locate/tests/colab/run_benchmark.py',
    '--runs','3','--fixture-dir',f'{ROOT}/beam-ranking-fixture'
], env=env, text=True)
if result.returncode not in (0,1):
    raise RuntimeError(f'benchmark runner failed with exit code {result.returncode}')


## Interpretation

A pass means the real FastContext model + native exploration loop + repo-locate wrapper recovered both required final-citation targets on the public fixture. This still does not test whether Codex, Claude Code, or Agy autonomously choose to invoke the skill.